In [0]:
import requests
import pandas as pd
import time
from pyspark.sql import SparkSession

def extrair_e_carregar_feriados():
    """
    EXTRACT & LOAD (ELT): Procura o histórico de feriados nacionais de 2011 até 2027
    usando a Brasil API e carrega os dados brutos na camada Bronze do Delta Lake.
    """
    # Histórico profundo alinhado com as cotações e clima!
    anos = list(range(2011, 2028))
    todos_feriados = []
    
    print(f"🔄 [Extract] A consumir dados de feriados para os anos: {anos}...")
    
    # Cabeçalho para evitar bloqueios de segurança (WAF)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    for ano in anos:
        url = f"https://brasilapi.com.br/api/feriados/v1/{ano}"
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 200:
                dados_ano = response.json()
                todos_feriados.extend(dados_ano)
            else:
                print(f"⚠️ Não foi possível obter feriados para o ano {ano}. Status: {response.status_code}")
            
            # Uma pequena pausa amigável para evitar rate limit na API pública
            time.sleep(0.5)
            
        except Exception as e:
            print(f"❌ Erro ao consultar o ano {ano}: {e}")
            
    if not todos_feriados:
        raise RuntimeError("❌ Nenhum dado de feriado foi retornado pela API.")
        
    # 1. Cria o DataFrame do Pandas com os dados acumulados
    df_pandas = pd.DataFrame(todos_feriados)
    
    # Garantimos que a data esteja no formato de string padrão (AAAA-MM-DD)
    df_pandas["date"] = pd.to_datetime(df_pandas["date"]).dt.strftime("%Y-%m-%d")
    
    # 2. Converte para Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    # 3. Garante que o schema bronze exista
    print("📁 Garantindo que o schema 'bronze' exista...")
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    
    # 4. [Load] Grava o dado bruto na tabela Bronze como Delta Lake
    print("💾 [Load] Salvando feriados brutos na tabela bronze.feriados_raw...")
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.feriados_raw")
        
    print("✅ Ingestão de Feriados concluída com sucesso!")

# Executa o pipeline de feriados
extrair_e_carregar_feriados()

In [0]:
%sql
SELECT * FROM bronze.feriados_raw ORDER BY date deSC;